# RetailMind AI 03: Feature Engineering

**Phase:** 3 — Feature Engineering

**Purpose:** Transform validated retail transaction data into ML-ready feature datasets.

**Outputs:**
- `data/processed/retail_cleaned.csv`
- `data/processed/daily_product_demand.csv`
- `data/processed/customer_features.csv`
- `data/processed/product_features.csv`

---
## Section 1 — Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
RAW_PATH = '../data/raw/Amazon.csv'
PROCESSED_DIR = '../data/processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)
print('Libraries loaded.')
print(f'Output dir: {os.path.abspath(PROCESSED_DIR)}')

Libraries loaded.
Output dir: D:\RetailMindAI\RetailMindAI\data\processed


---
## Section 2 — Load Data

Load raw dataset and verify all 20 required columns exist.

> `data/raw/Amazon.csv` is **never modified**. All work is done on copies.

In [2]:
df_raw = pd.read_csv(RAW_PATH)
print(f'Loaded: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
REQUIRED_COLUMNS = ['OrderID','OrderDate','CustomerID','CustomerName','ProductID','ProductName','Category','Brand','Quantity','UnitPrice','Discount','Tax','ShippingCost','TotalAmount','PaymentMethod','OrderStatus','City','State','Country','SellerID']
missing_cols = [c for c in REQUIRED_COLUMNS if c not in df_raw.columns]
print(f'Missing cols: {len(missing_cols)}')
if missing_cols:
    raise ValueError(f'STOPPING: Missing columns: {missing_cols}')
else:
    print('All 20 required columns present.')

Loaded: 100,000 rows x 20 columns
Missing cols: 0
All 20 required columns present.


---
## Section 3 — Create Clean Transaction Table

### Cleaning decisions (from 01_data_understanding.ipynb)

| Finding | Action |
|---|---|
| `OrderDate` stored as string | Convert to datetime64 |
| `Discount` is fractional (0.0-0.30) | Retain as-is |
| Zero Tax on 9,878 rows | Retain - tax-exempt orders are valid |
| Zero Discount on 40,246 rows | Retain - no-discount orders are valid |
| Zero ShippingCost on 20 rows | Retain - free shipping is valid |
| Returned / Cancelled orders | **Retain all** - OrderStatus is business information |
| No missing values | No imputation required |
| No duplicate OrderIDs | No deduplication required |

> Cancelled and returned orders are NOT deleted. They contribute to return/cancellation rates and future conflict signals.

In [3]:
df = df_raw.copy()
df['OrderDate'] = pd.to_datetime(df['OrderDate'], errors='coerce')
print(f'Invalid dates: {df["OrderDate"].isna().sum()}')
df['is_returned']  = (df['OrderStatus'] == 'Returned').astype(int)
df['is_cancelled'] = (df['OrderStatus'] == 'Cancelled').astype(int)
df['is_delivered'] = (df['OrderStatus'] == 'Delivered').astype(int)
df['is_shipped']   = (df['OrderStatus'] == 'Shipped').astype(int)
df['is_pending']   = (df['OrderStatus'] == 'Pending').astype(int)
df['DiscountPct']  = df['Discount'] * 100
print(f'Clean table shape: {df.shape}')
print(f'Date range: {df["OrderDate"].min().date()} to {df["OrderDate"].max().date()}')
print(df['OrderStatus'].value_counts())

Invalid dates: 0
Clean table shape: (100000, 26)
Date range: 2020-01-01 to 2024-12-29
OrderStatus
Delivered    74628
Shipped      15192
Pending       4103
Returned      3049
Cancelled     3028
Name: count, dtype: int64


In [4]:
RETAIL_CLEANED_PATH = os.path.join(PROCESSED_DIR, 'retail_cleaned.csv')
df.to_csv(RETAIL_CLEANED_PATH, index=False)
print(f'Saved: {RETAIL_CLEANED_PATH}')
print(f'Shape: {df.shape}')

Saved: ../data/processed\retail_cleaned.csv
Shape: (100000, 26)


---
## Section 4 — Daily Product Demand Dataset

**Purpose:** Aggregate to `Date x ProductID` granularity. Forecasting target = `DailyQuantity`.

> **Why not TotalAmount?** Revenue is influenced by discount, tax, shipping. Quantity is the direct, clean demand signal.

### Data Discovery: ProductID vs Category/Brand

> **Important finding:** In this dataset, each `ProductID` appears under multiple `Category` and `Brand` values per order. Grouping by `[Date, ProductID, ProductName, Category, Brand]` would create duplicate (ProductID, Date) pairs. The correct aggregation groups by `[Date, ProductID]` only.

**Handling of Returned/Cancelled orders:** Retained. `DailyQuantity` covers ALL order statuses. Return/cancellation counts are computed separately to preserve business meaning.

In [5]:
df['Date'] = pd.to_datetime(df['OrderDate'].dt.date)

# Aggregate by [Date, ProductID] ONLY
# NOTE: Category and Brand are intentionally excluded from the groupby key.
# Each ProductID appears in multiple Category/Brand combinations in this
# synthetic dataset, so including them would create duplicate (ProductID, Date) rows.
daily_agg = df.groupby(['Date', 'ProductID']).agg(
    DailyQuantity          = ('Quantity',     'sum'),
    DailyRevenue           = ('TotalAmount',  'sum'),
    DailyOrderCount        = ('OrderID',      'count'),
    DailyAveragePrice      = ('UnitPrice',    'mean'),
    DailyAverageDiscount   = ('Discount',     'mean'),
    DailyReturnCount       = ('is_returned',  'sum'),
    DailyCancellationCount = ('is_cancelled', 'sum'),
).reset_index()

daily_agg['DailyAverageOrderValue'] = daily_agg['DailyRevenue'] / daily_agg['DailyOrderCount']
daily_agg = daily_agg.sort_values(['ProductID','Date']).reset_index(drop=True)

print(f'Daily demand shape: {daily_agg.shape}')
print(f'Duplicates (ProductID, Date): {daily_agg.duplicated(["ProductID","Date"]).sum()} (expected 0)')
print(f'Products: {daily_agg["ProductID"].nunique()} | Dates: {daily_agg["Date"].nunique()}')
daily_agg.head(8)

Daily demand shape: (60649, 10)
Duplicates (ProductID, Date): 0 (expected 0)
Products: 50 | Dates: 1825


,Date,ProductID,DailyQuantity,DailyRevenue,DailyOrderCount,DailyAveragePrice,DailyAverageDiscount,DailyReturnCount,DailyCancellationCount,DailyAverageOrderValue
0,2020-01-01,P00001,2,1066.4300,1,530.0100,0.0500,0,0,1066.4300
1,2020-01-03,P00001,5,1939.8400,1,491.9400,0.2500,0,0,1939.8400
2,2020-01-04,P00001,3,972.0000,2,232.8300,0.0000,0,0,486.0000
3,2020-01-05,P00001,1,366.0300,1,355.6600,0.0500,0,0,366.0300
4,2020-01-06,P00001,2,1151.3500,1,512.7300,0.0000,0,0,1151.3500
5,2020-01-08,P00001,7,2612.5100,2,303.6450,0.0250,0,0,1306.2550
6,2020-01-09,P00001,3,1206.4400,2,412.2450,0.0750,0,1,603.2200
7,2020-01-10,P00001,5,704.5000,1,139.9600,0.0500,0,0,704.5000


---
## Section 5 — Complete Product-Date Grid

### Decision on gap filling

Missing product-date cells could mean **zero demand** or **missing observation**. This distinction is critical for forecasting.

**Investigation and documented assumption follow.**

In [6]:
all_dates    = pd.date_range(daily_agg['Date'].min(), daily_agg['Date'].max(), freq='D')
all_products = daily_agg['ProductID'].unique()

total_possible  = len(all_dates) * len(all_products)
actual_observed = len(daily_agg)
missing_cells   = total_possible - actual_observed
pct_missing     = missing_cells / total_possible * 100

print(f'Total possible cells : {total_possible:,}')
print(f'Observed cells       : {actual_observed:,}')
print(f'Missing cells        : {missing_cells:,}')
print(f'Grid density         : {100 - pct_missing:.1f}%')

per_product = daily_agg.groupby('ProductID')['Date'].count()
print(f'Dates per product: min={per_product.min()}, max={per_product.max()}, mean={per_product.mean():.0f}')

Total possible cells : 91,250
Observed cells       : 60,649
Missing cells        : 30,601
Grid density         : 66.5%
Dates per product: min=1169, max=1248, mean=1213


In [7]:
print('ASSUMPTION: Missing product-date cells = ZERO OBSERVED DEMAND.')
print('Rationale:')
print('  All 50 products span the full 2020-2024 date range.')
print('  ~1.1 orders/product/day on average - sparse but active throughout.')
print('  Zero-filling is required for lag/rolling features to be valid.')
print('  NOTE: Revisit if product launch dates become available.')

# Product metadata: consistent ProductName per ProductID
product_meta = df.groupby('ProductID')['ProductName'].first().reset_index()

# Build full Cartesian product of ProductID x Date
full_index = pd.MultiIndex.from_product(
    [all_products, all_dates], names=['ProductID', 'Date']
)
demand_full = pd.DataFrame(index=full_index).reset_index()
demand_full = demand_full.merge(daily_agg, on=['ProductID', 'Date'], how='left')
demand_full = demand_full.merge(product_meta, on='ProductID', how='left')

fill_cols = [
    'DailyQuantity', 'DailyRevenue', 'DailyOrderCount',
    'DailyAveragePrice', 'DailyAverageDiscount',
    'DailyReturnCount', 'DailyCancellationCount', 'DailyAverageOrderValue'
]
for col in fill_cols:
    demand_full[col] = demand_full[col].fillna(0)

demand_full = demand_full.sort_values(['ProductID', 'Date']).reset_index(drop=True)
print(f'Full grid: {demand_full.shape}  |  Expected: {len(all_dates)*len(all_products):,}')
print(f'Duplicates: {demand_full.duplicated(["ProductID","Date"]).sum()} (expected 0)')

ASSUMPTION: Missing product-date cells = ZERO OBSERVED DEMAND.
Rationale:
  All 50 products span the full 2020-2024 date range.
  ~1.1 orders/product/day on average - sparse but active throughout.
  Zero-filling is required for lag/rolling features to be valid.
  NOTE: Revisit if product launch dates become available.


Full grid: (91250, 11)  |  Expected: 91,250
Duplicates: 0 (expected 0)


---
## Section 6 — Temporal Features

Calendar features for seasonal patterns. No leakage — derived from Date column only.

In [8]:
demand_full['year']            = demand_full['Date'].dt.year
demand_full['month']           = demand_full['Date'].dt.month_name()
demand_full['month_number']    = demand_full['Date'].dt.month
demand_full['quarter']         = demand_full['Date'].dt.quarter
demand_full['week_of_year']    = demand_full['Date'].dt.isocalendar().week.astype(int)
demand_full['day_of_week']     = demand_full['Date'].dt.day_name()
demand_full['day_of_week_num'] = demand_full['Date'].dt.dayofweek
demand_full['day_of_month']    = demand_full['Date'].dt.day
demand_full['is_weekend']      = (demand_full['Date'].dt.dayofweek >= 5).astype(int)
demand_full['is_month_start']  = demand_full['Date'].dt.is_month_start.astype(int)
demand_full['is_month_end']    = demand_full['Date'].dt.is_month_end.astype(int)
temporal_features = ['year','month','month_number','quarter','week_of_year','day_of_week','day_of_week_num','day_of_month','is_weekend','is_month_start','is_month_end']
print(f'Temporal features created: {len(temporal_features)}')
demand_full[['Date']+temporal_features].drop_duplicates('Date').head(10)

Temporal features created: 11


,Date,year,month,month_number,quarter,week_of_year,day_of_week,day_of_week_num,day_of_month,is_weekend,is_month_start,is_month_end
0,2020-01-01,2020,January,1,1,1,Wednesday,2,1,0,1,0
1,2020-01-02,2020,January,1,1,1,Thursday,3,2,0,0,0
2,2020-01-03,2020,January,1,1,1,Friday,4,3,0,0,0
3,2020-01-04,2020,January,1,1,1,Saturday,5,4,1,0,0
4,2020-01-05,2020,January,1,1,1,Sunday,6,5,1,0,0
5,2020-01-06,2020,January,1,1,2,Monday,0,6,0,0,0
6,2020-01-07,2020,January,1,1,2,Tuesday,1,7,0,0,0
7,2020-01-08,2020,January,1,1,2,Wednesday,2,8,0,0,0
8,2020-01-09,2020,January,1,1,2,Thursday,3,9,0,0,0
9,2020-01-10,2020,January,1,1,2,Friday,4,10,0,0,0


---
## Section 7 — Demand Lag Features

**Definition:** `lag_N` at date `t` = DailyQuantity at `t-N` for the **same product**.

**Leakage prevention:** `groupby(ProductID).shift(N)` — no cross-product contamination.

**Expected NaN:** First N days per product for each lag. This is correct, not a data error.

In [9]:
demand_full = demand_full.sort_values(['ProductID','Date']).reset_index(drop=True)
for lag in [1, 7, 14, 28]:
    demand_full[f'lag_{lag}'] = demand_full.groupby('ProductID')['DailyQuantity'].shift(lag)
lag_features = ['lag_1','lag_7','lag_14','lag_28']
print('Lag features created:', lag_features)
print()
print('NaN counts per lag (expected):')
for col in lag_features:
    n = demand_full[col].isna().sum()
    print(f'  {col}: {n:,} NaN ({n/len(demand_full)*100:.2f}%)')
print()
sp = demand_full['ProductID'].iloc[0]
demand_full[demand_full['ProductID']==sp][['Date','DailyQuantity']+lag_features].head(32)

Lag features created: ['lag_1', 'lag_7', 'lag_14', 'lag_28']

NaN counts per lag (expected):


  lag_1: 50 NaN (0.05%)
  lag_7: 350 NaN (0.38%)
  lag_14: 700 NaN (0.77%)
  lag_28: 1,400 NaN (1.53%)



,Date,DailyQuantity,lag_1,lag_7,lag_14,lag_28
0,2020-01-01,2.0000,NaN,NaN,NaN,NaN
1,2020-01-02,0.0000,2.0000,NaN,NaN,NaN
2,2020-01-03,5.0000,0.0000,NaN,NaN,NaN
3,2020-01-04,3.0000,5.0000,NaN,NaN,NaN
4,2020-01-05,1.0000,3.0000,NaN,NaN,NaN
5,2020-01-06,2.0000,1.0000,NaN,NaN,NaN
6,2020-01-07,0.0000,2.0000,NaN,NaN,NaN
7,2020-01-08,7.0000,0.0000,2.0000,NaN,NaN
8,2020-01-09,3.0000,7.0000,0.0000,NaN,NaN
9,2020-01-10,5.0000,3.0000,5.0000,NaN,NaN


---
## Section 8 — Rolling Demand Features

**Leakage prevention:** `shift(1)` before `.rolling(window)` ensures current-day demand is excluded.

- `rolling_mean_7` at `t` = mean of DailyQuantity from `t-7` to `t-1`

**Expected NaN:** First N rows per product for window size N.

In [10]:
demand_full = demand_full.sort_values(['ProductID','Date']).reset_index(drop=True)
for w in [7, 14, 28]:
    shifted = demand_full.groupby('ProductID')['DailyQuantity'].shift(1)
    demand_full[f'rolling_mean_{w}'] = shifted.groupby(demand_full['ProductID']).transform(
        lambda x: x.rolling(window=w, min_periods=w).mean())
    demand_full[f'rolling_std_{w}'] = shifted.groupby(demand_full['ProductID']).transform(
        lambda x: x.rolling(window=w, min_periods=w).std())
rolling_features = ['rolling_mean_7','rolling_mean_14','rolling_mean_28','rolling_std_7','rolling_std_14','rolling_std_28']
print(f'Rolling features: {len(rolling_features)}')
for col in rolling_features:
    n=demand_full[col].isna().sum()
    print(f'  {col}: {n:,} NaN ({n/len(demand_full)*100:.2f}%)')

Rolling features: 6
  rolling_mean_7: 350 NaN (0.38%)
  rolling_mean_14: 700 NaN (0.77%)
  rolling_mean_28: 1,400 NaN (1.53%)
  rolling_std_7: 350 NaN (0.38%)
  rolling_std_14: 700 NaN (0.77%)
  rolling_std_28: 1,400 NaN (1.53%)


---
## Section 9 — Demand Trend Features

| Feature | Description |
|---|---|
| `short_term_mean` | 7-day historical rolling mean |
| `medium_term_mean` | 28-day historical rolling mean |
| `long_term_mean` | 90-day historical rolling mean |
| `short_vs_medium_growth` | `short/medium - 1` |
| `medium_vs_long_growth` | `medium/long - 1` |

**Conflict readiness:** `short_vs_medium_growth > 0` AND `medium_vs_long_growth < 0` = short-term spike on long-term downward trend (Temporal Conflict).

> Only numerical features here. Conflict logic is Phase 4+.

In [11]:
def grouped_shifted_rolling(df, group_col, value_col, shift_n, window, agg='mean'):
    '''Compute grouped shift then rolling aggregate. Prevents leakage.'''
    shifted = df.groupby(group_col)[value_col].shift(shift_n)
    if agg == 'mean':
        return shifted.groupby(df[group_col]).transform(lambda x: x.rolling(window=window, min_periods=window).mean())
    elif agg == 'std':
        return shifted.groupby(df[group_col]).transform(lambda x: x.rolling(window=window, min_periods=window).std())

demand_full['short_term_mean']  = grouped_shifted_rolling(demand_full,'ProductID','DailyQuantity',1,7)
demand_full['medium_term_mean'] = grouped_shifted_rolling(demand_full,'ProductID','DailyQuantity',1,28)
demand_full['long_term_mean']   = grouped_shifted_rolling(demand_full,'ProductID','DailyQuantity',1,90)
demand_full['short_vs_medium_growth'] = np.where(demand_full['medium_term_mean'] > 0,
    demand_full['short_term_mean']/demand_full['medium_term_mean'] - 1, np.nan)
demand_full['medium_vs_long_growth']  = np.where(demand_full['long_term_mean'] > 0,
    demand_full['medium_term_mean']/demand_full['long_term_mean'] - 1, np.nan)
trend_features = ['short_term_mean','medium_term_mean','long_term_mean','short_vs_medium_growth','medium_vs_long_growth']
print(f'Trend features: {len(trend_features)}')
for col in trend_features:
    n=demand_full[col].isna().sum()
    print(f'  {col}: {n:,} NaN ({n/len(demand_full)*100:.2f}%)')

Trend features: 5
  short_term_mean: 350 NaN (0.38%)
  medium_term_mean: 1,400 NaN (1.53%)
  long_term_mean: 4,500 NaN (4.93%)
  short_vs_medium_growth: 1,400 NaN (1.53%)
  medium_vs_long_growth: 4,500 NaN (4.93%)


---
## Section 10 — Demand Volatility Features

| Feature | Formula | Interpretation |
|---|---|---|
| `demand_std_7` | Rolling std 7d (shift=1) | Short-term volatility |
| `demand_std_28` | Rolling std 28d (shift=1) | Medium-term volatility |
| `cv_7` | `demand_std_7 / rolling_mean_7` | Coefficient of Variation (7d) |
| `cv_28` | `demand_std_28 / rolling_mean_28` | Coefficient of Variation (28d) |

**CV:** `sigma/mu`. CV > 1 = highly volatile relative to mean.

In [12]:
demand_full['demand_std_7']  = demand_full['rolling_std_7']
demand_full['demand_std_28'] = demand_full['rolling_std_28']
demand_full['cv_7']  = np.where(demand_full['rolling_mean_7']  > 0, demand_full['demand_std_7']  / demand_full['rolling_mean_7'],  np.nan)
demand_full['cv_28'] = np.where(demand_full['rolling_mean_28'] > 0, demand_full['demand_std_28'] / demand_full['rolling_mean_28'], np.nan)
volatility_features = ['demand_std_7','demand_std_28','cv_7','cv_28']
print(f'Volatility features: {len(volatility_features)}')
demand_full[volatility_features].describe()

Volatility features: 4


,demand_std_7,demand_std_28,cv_7,cv_28
count,90900.0000,89850.0000,90854.0000,89850.0000
mean,3.2670,3.4184,1.0779,1.0586
std,1.1675,0.6025,0.3620,0.1749
min,0.0000,1.5477,0.1207,0.5236
25%,2.4103,2.9921,0.8273,0.9350
50%,3.1320,3.3798,1.0281,1.0443
75%,3.9461,3.7957,1.2689,1.1639
max,10.8277,6.1673,2.6458,2.2587


---
## Section 11 — Return and Cancellation Signals

- `daily_return_rate` = `DailyReturnCount / DailyOrderCount`
- `rolling_return_rate_28` = 28-day historical rolling mean (shift=1)
- `daily_cancel_rate` = `DailyCancellationCount / DailyOrderCount`
- `rolling_cancel_rate_28` = 28-day historical rolling mean (shift=1)

In [13]:
demand_full['daily_return_rate'] = np.where(demand_full['DailyOrderCount']>0,
    demand_full['DailyReturnCount']/demand_full['DailyOrderCount'], 0)
demand_full['daily_cancel_rate'] = np.where(demand_full['DailyOrderCount']>0,
    demand_full['DailyCancellationCount']/demand_full['DailyOrderCount'], 0)
demand_full['rolling_return_rate_28'] = grouped_shifted_rolling(demand_full,'ProductID','daily_return_rate',1,28)
demand_full['rolling_cancel_rate_28'] = grouped_shifted_rolling(demand_full,'ProductID','daily_cancel_rate',1,28)
signal_features = ['daily_return_rate','daily_cancel_rate','rolling_return_rate_28','rolling_cancel_rate_28']
print(f'Signal features: {len(signal_features)}')
demand_full[signal_features].describe()

Signal features: 4


,daily_return_rate,daily_cancel_rate,rolling_return_rate_28,rolling_cancel_rate_28
count,91250.0000,91250.0000,89850.0000,89850.0000
mean,0.0202,0.0199,0.0202,0.0198
std,0.1216,0.1199,0.0232,0.0227
min,0.0000,0.0000,0.0000,0.0000
25%,0.0000,0.0000,0.0000,0.0000
50%,0.0000,0.0000,0.0179,0.0119
75%,0.0000,0.0000,0.0357,0.0357
max,1.0000,1.0000,0.1875,0.1607


---
## Section 12 — Revenue Features

**Purpose:** Revenue is a business signal, NOT the forecasting target.

> `TotalAmount` is NOT predicted. Revenue supports conflict detection only.

In [14]:
demand_full['rolling_revenue_mean_28'] = grouped_shifted_rolling(demand_full,'ProductID','DailyRevenue',1,28)
demand_full['rolling_revenue_std_28']  = grouped_shifted_rolling(demand_full,'ProductID','DailyRevenue',1,28,agg='std')
rv7  = grouped_shifted_rolling(demand_full,'ProductID','DailyRevenue',1,7)
rv28 = demand_full['rolling_revenue_mean_28']
demand_full['revenue_growth_7_vs_28'] = np.where(rv28 > 0, rv7/rv28 - 1, np.nan)
revenue_features = ['DailyRevenue','DailyAverageOrderValue','rolling_revenue_mean_28','rolling_revenue_std_28','revenue_growth_7_vs_28']
print(f'Revenue features: {len(revenue_features)}')
demand_full[revenue_features].describe()

Revenue features: 5


,DailyRevenue,DailyAverageOrderValue,rolling_revenue_mean_28,rolling_revenue_std_28,revenue_growth_7_vs_28
count,91250.0000,91250.0000,89850.0000,89850.0000,89850.0000
mean,1006.3085,610.1295,1006.1292,1198.4366,-0.0002
std,1223.0257,668.9531,228.1033,242.4644,0.4036
min,0.0000,0.0000,293.6579,456.2736,-1.0000
25%,0.0000,0.0000,846.6568,1030.0341,-0.2862
50%,549.1250,432.2775,998.3536,1181.2557,-0.0239
75%,1648.1300,1015.9700,1156.0811,1350.2308,0.2617
max,10734.3400,3514.2400,2224.8168,2336.9412,1.8260


---
## Section 13 — Save Daily Product Demand Dataset

In [15]:
DEMAND_PATH = os.path.join(PROCESSED_DIR, 'daily_product_demand.csv')
demand_full.to_csv(DEMAND_PATH, index=False)
print(f'Saved: {DEMAND_PATH}')
print(f'Shape: {demand_full.shape[0]:,} rows x {demand_full.shape[1]} columns')
for i, col in enumerate(demand_full.columns, 1):
    print(f'  {i:2d}. {col}')

Saved: ../data/processed\daily_product_demand.csv
Shape: 91,250 rows x 48 columns
   1. ProductID
   2. Date
   3. DailyQuantity
   4. DailyRevenue
   5. DailyOrderCount
   6. DailyAveragePrice
   7. DailyAverageDiscount
   8. DailyReturnCount
   9. DailyCancellationCount
  10. DailyAverageOrderValue
  11. ProductName
  12. year
  13. month
  14. month_number
  15. quarter
  16. week_of_year
  17. day_of_week
  18. day_of_week_num
  19. day_of_month
  20. is_weekend
  21. is_month_start
  22. is_month_end
  23. lag_1
  24. lag_7
  25. lag_14
  26. lag_28
  27. rolling_mean_7
  28. rolling_std_7
  29. rolling_mean_14
  30. rolling_std_14
  31. rolling_mean_28
  32. rolling_std_28
  33. short_term_mean
  34. medium_term_mean
  35. long_term_mean
  36. short_vs_medium_growth
  37. medium_vs_long_growth
  38. demand_std_7
  39. demand_std_28
  40. cv_7
  41. cv_28
  42. daily_return_rate
  43. daily_cancel_rate
  44. rolling_return_rate_28
  45. rolling_cancel_rate_28
  46. rolling_revenue

---
## Section 14 — Customer-Level Features

**RFM Definition:**
- **Recency:** Days since last order (from reference date 2024-12-29)
- **Frequency:** Total orders placed
- **Monetary:** Total amount spent

**Reference date:** `2024-12-29` (last date in dataset — reproducible)

> No K-Means here. Segment assignment is Phase 4.

In [16]:
REFERENCE_DATE = pd.Timestamp('2024-12-29')
print(f'RFM Reference date: {REFERENCE_DATE.date()} (last date in dataset)')
cust_base = df.groupby('CustomerID').agg(
    last_order_date=('OrderDate','max'), first_order_date=('OrderDate','min'),
    Frequency=('OrderID','count'), Monetary=('TotalAmount','sum'),
    TotalQuantity=('Quantity','sum'), UniqueProducts=('ProductID','nunique'),
    UniqueCategories=('Category','nunique'), TotalReturns=('is_returned','sum'),
    TotalCancellations=('is_cancelled','sum'), TotalDiscount=('Discount','sum'),
).reset_index()
cust_base['Recency']              = (REFERENCE_DATE - cust_base['last_order_date']).dt.days
cust_base['AverageOrderValue']    = cust_base['Monetary'] / cust_base['Frequency']
cust_base['ReturnRate']           = cust_base['TotalReturns'] / cust_base['Frequency']
cust_base['CancellationRate']     = cust_base['TotalCancellations'] / cust_base['Frequency']
cust_base['AverageDiscount']      = cust_base['TotalDiscount'] / cust_base['Frequency']
cust_base['ActiveDays']           = (cust_base['last_order_date'] - cust_base['first_order_date']).dt.days + 1
cust_base['AverageOrderFrequency']= cust_base['Frequency'] / cust_base['ActiveDays']
customer_features = cust_base[['CustomerID','Recency','Frequency','Monetary','TotalQuantity','AverageOrderValue','UniqueProducts','UniqueCategories','ReturnRate','CancellationRate','AverageDiscount','AverageOrderFrequency','first_order_date','last_order_date','ActiveDays']].copy()
print(f'Customer features: {customer_features.shape}')
customer_features.describe()

RFM Reference date: 2024-12-29 (last date in dataset)
Customer features: (43233, 15)


,Recency,Frequency,Monetary,TotalQuantity,AverageOrderValue,UniqueProducts,UniqueCategories,ReturnRate,CancellationRate,AverageDiscount,AverageOrderFrequency,first_order_date,last_order_date,ActiveDays
count,43233.0000,43233.0000,43233.0000,43233.0000,43233.0000,43233.0000,43233.0000,43233.0000,43233.0000,43233.0000,43233.0000,43233,43233,43233.0000
mean,627.1655,2.3130,2123.9712,6.9424,918.4592,2.2700,1.9701,0.0299,0.0304,0.0742,0.3193,2021-09-18 15:07:16.416626,2023-04-11 20:01:40.922906,571.2045
min,0.0000,1.0000,6.1900,1.0000,6.1900,1.0000,1.0000,0.0000,0.0000,0.0000,0.0011,2020-01-01 00:00:00,2020-01-01 00:00:00,1.0000
25%,224.0000,1.0000,876.1900,4.0000,508.9400,1.0000,1.0000,0.0000,0.0000,0.0250,0.0030,2020-08-09 00:00:00,2022-05-21 00:00:00,1.0000
50%,518.0000,2.0000,1805.4100,6.0000,837.0933,2.0000,2.0000,0.0000,0.0000,0.0625,0.0055,2021-06-02 00:00:00,2023-07-30 00:00:00,481.0000
75%,953.0000,3.0000,2992.4300,9.0000,1226.1467,3.0000,3.0000,0.0000,0.0000,0.1000,1.0000,2022-08-12 00:00:00,2024-05-19 00:00:00,1036.0000
max,1824.0000,10.0000,15215.7900,35.0000,3484.4400,10.0000,6.0000,1.0000,1.0000,0.3000,2.0000,2024-12-29 00:00:00,2024-12-29 00:00:00,1820.0000
std,478.1221,1.2571,1594.3794,4.3479,551.5907,1.2174,0.9430,0.1280,0.1302,0.0627,0.4609,NaN,NaN,544.4199


In [17]:
CUSTOMER_PATH = os.path.join(PROCESSED_DIR, 'customer_features.csv')
customer_features.to_csv(CUSTOMER_PATH, index=False)
print(f'Saved: {CUSTOMER_PATH} | Shape: {customer_features.shape}')

Saved: ../data/processed\customer_features.csv | Shape: (43233, 15)


---
## Section 15 — Product-Level Features

### Data Discovery

> **Important finding:** In this dataset, `ProductID` is NOT uniquely associated with a single `Category` or `Brand`. Each product appears under multiple categories and brands (a characteristic of synthetic Amazon data).

**Implication:** Grouping by `[ProductID, ProductName, Category, Brand]` creates a cross-product of 3,000 rows for 50 products. The correct approach is to aggregate by `ProductID` only.

| Feature | Definition |
|---|---|
| `UniqueCategories` | Number of distinct categories product appears in |
| `UniqueBrands` | Number of distinct brands product appears in |
| `RecentDemand` | Total qty sold in last 90 days |
| `HistoricalAverageDemand` | Mean daily demand over full history |
| `DemandVolatility` | Std dev of daily demand over full history |
| `DemandGrowth` | `recent_daily_avg / historical_avg - 1` |

In [18]:
REFERENCE_DATE = pd.Timestamp('2024-12-29')
RECENT_CUTOFF  = REFERENCE_DATE - pd.Timedelta(days=90)

# Step 1: Product metadata (consistent name per ProductID)
product_meta = df.groupby('ProductID').agg(
    ProductName = ('ProductName', 'first'),
).reset_index()

# Step 2: Aggregate all metrics by ProductID only
# NOTE: Category and Brand intentionally excluded from groupby key
# because this dataset assigns a product to multiple categories/brands
prod_base = df.groupby('ProductID').agg(
    TotalQuantity      = ('Quantity',     'sum'),
    TotalRevenue       = ('TotalAmount',  'sum'),
    OrderCount         = ('OrderID',      'count'),
    TotalReturns       = ('is_returned',  'sum'),
    TotalCancellations = ('is_cancelled', 'sum'),
    UniqueCustomers    = ('CustomerID',   'nunique'),
    UniqueCategories   = ('Category',     'nunique'),
    UniqueBrands       = ('Brand',        'nunique'),
    AveragePrice       = ('UnitPrice',    'mean'),
    AverageDiscount    = ('Discount',     'mean'),
).reset_index()

prod_base['AverageOrderValue'] = prod_base['TotalRevenue'] / prod_base['OrderCount']
prod_base['ReturnRate']        = prod_base['TotalReturns'] / prod_base['OrderCount']
prod_base['CancellationRate']  = prod_base['TotalCancellations'] / prod_base['OrderCount']

# Step 3: Recent demand (last 90 days)
recent_qty = (
    df[df['OrderDate'] >= RECENT_CUTOFF]
    .groupby('ProductID')['Quantity'].sum().reset_index()
    .rename(columns={'Quantity': 'RecentDemand'})
)
prod_base = prod_base.merge(recent_qty, on='ProductID', how='left')
prod_base['RecentDemand'] = prod_base['RecentDemand'].fillna(0)

# Step 4: Historical stats from full demand grid
hist_stats = demand_full.groupby('ProductID')['DailyQuantity'].agg(
    HistoricalAverageDemand='mean',
    DemandVolatility='std'
).reset_index()
prod_base = prod_base.merge(hist_stats, on='ProductID', how='left')

# Step 5: Demand growth ratio
prod_base['RecentDailyAvg'] = prod_base['RecentDemand'] / 90
prod_base['DemandGrowth']   = np.where(
    prod_base['HistoricalAverageDemand'] > 0,
    prod_base['RecentDailyAvg'] / prod_base['HistoricalAverageDemand'] - 1, np.nan
)

# Step 6: Merge product metadata
prod_base = prod_base.merge(product_meta, on='ProductID', how='left')

product_features = prod_base[[
    'ProductID', 'ProductName',
    'TotalQuantity', 'TotalRevenue', 'OrderCount',
    'AverageOrderValue', 'AveragePrice', 'AverageDiscount',
    'ReturnRate', 'CancellationRate',
    'UniqueCustomers', 'UniqueCategories', 'UniqueBrands',
    'RecentDemand', 'HistoricalAverageDemand',
    'DemandVolatility', 'DemandGrowth'
]].copy()

print(f'Product features shape: {product_features.shape}')
print(f'Unique ProductIDs      : {product_features["ProductID"].nunique()} (expected 50)')
product_features.head(10)

Product features shape: (50, 17)
Unique ProductIDs      : 50 (expected 50)


,ProductID,ProductName,TotalQuantity,TotalRevenue,OrderCount,AverageOrderValue,AveragePrice,AverageDiscount,ReturnRate,CancellationRate,UniqueCustomers,UniqueCategories,UniqueBrands,RecentDemand,HistoricalAverageDemand,DemandVolatility,DemandGrowth
0,P00001,Wireless Earbuds,6008,1834207.5800,2008,913.4500,300.8321,0.0776,0.0324,0.0284,1982,6,10,279,3.2921,3.5154,-0.0583
1,P00002,Bluetooth Speaker,5782,1803356.7600,1913,942.6852,310.9409,0.0748,0.0220,0.0282,1880,6,10,338,3.1682,3.4211,0.1854
2,P00003,Smartphone Case,6060,1875421.4100,1989,942.8966,307.1541,0.0730,0.0297,0.0287,1955,6,10,289,3.3205,3.4816,-0.0330
3,P00004,USB-C Charger,5811,1763075.2600,1958,900.4470,303.0005,0.0806,0.0317,0.0337,1930,6,10,285,3.1841,3.3487,-0.0055
4,P00005,Laptop Sleeve,5992,1848297.8000,1996,926.0009,304.3442,0.0709,0.0296,0.0230,1953,6,10,258,3.2833,3.5169,-0.1269
5,P00006,Gaming Mouse,6170,1895103.9800,2010,942.8378,303.3702,0.0733,0.0333,0.0378,1975,6,10,256,3.3808,3.4187,-0.1587
6,P00007,Mechanical Keyboard,6161,1906963.5400,2034,937.5435,306.5985,0.0750,0.0310,0.0324,1991,6,10,331,3.3759,3.6110,0.0894
7,P00008,4K Monitor,6111,1866774.3500,2006,930.5954,298.1829,0.0748,0.0284,0.0389,1976,6,10,358,3.3485,3.5741,0.1879
8,P00009,Portable SSD 1TB,5996,1819377.3800,2030,896.2450,300.1726,0.0750,0.0300,0.0256,1986,6,10,285,3.2855,3.4312,-0.0362
9,P00010,Smartwatch,5983,1901275.5900,1989,955.8952,312.7875,0.0721,0.0342,0.0266,1946,6,10,247,3.2784,3.3953,-0.1629


In [19]:
PRODUCT_PATH = os.path.join(PROCESSED_DIR, 'product_features.csv')
product_features.to_csv(PRODUCT_PATH, index=False)
print(f'Saved: {PRODUCT_PATH}')
print(f'Shape: {product_features.shape[0]:,} rows x {product_features.shape[1]} columns')

Saved: ../data/processed\product_features.csv
Shape: 50 rows x 17 columns


---
## Section 16 — Customer-Product Interaction Features

Adds repeat-customer signals to the product feature set. These features support detection of the **commercial conflict**: revenue rising while unique customer count falls (few repeat buyers driving all revenue).

In [20]:
repeat_df = (
    df.groupby(['ProductID', 'CustomerID'])['OrderID']
    .count().reset_index()
    .rename(columns={'OrderID': 'orders_per_customer'})
)
repeat_df['is_repeat'] = (repeat_df['orders_per_customer'] > 1).astype(int)

repeat_agg = repeat_df.groupby('ProductID').agg(
    repeat_customers = ('is_repeat',   'sum'),
    total_cust_seen  = ('CustomerID',  'count'),
).reset_index()
repeat_agg['repeat_customer_ratio'] = (
    repeat_agg['repeat_customers'] / repeat_agg['total_cust_seen']
)

avg_freq = (
    repeat_df.groupby('ProductID')['orders_per_customer']
    .mean().reset_index()
    .rename(columns={'orders_per_customer': 'avg_customer_frequency_for_product'})
)

product_features = product_features.merge(
    repeat_agg[['ProductID', 'repeat_customers', 'repeat_customer_ratio']],
    on='ProductID', how='left'
)
product_features = product_features.merge(avg_freq, on='ProductID', how='left')

interaction_features = ['repeat_customers', 'repeat_customer_ratio',
                        'avg_customer_frequency_for_product']
print(f'Interaction features added: {interaction_features}')
print(f'Final product_features shape: {product_features.shape}')
product_features[['ProductID','ProductName','UniqueCustomers'] + interaction_features].head(10)

Interaction features added: ['repeat_customers', 'repeat_customer_ratio', 'avg_customer_frequency_for_product']
Final product_features shape: (50, 20)


,ProductID,ProductName,UniqueCustomers,repeat_customers,repeat_customer_ratio,avg_customer_frequency_for_product
0,P00001,Wireless Earbuds,1982,25,0.0126,1.0131
1,P00002,Bluetooth Speaker,1880,32,0.0170,1.0176
2,P00003,Smartphone Case,1955,34,0.0174,1.0174
3,P00004,USB-C Charger,1930,28,0.0145,1.0145
4,P00005,Laptop Sleeve,1953,43,0.0220,1.0220
5,P00006,Gaming Mouse,1975,35,0.0177,1.0177
6,P00007,Mechanical Keyboard,1991,43,0.0216,1.0216
7,P00008,4K Monitor,1976,30,0.0152,1.0152
8,P00009,Portable SSD 1TB,1986,44,0.0222,1.0222
9,P00010,Smartwatch,1946,41,0.0211,1.0221


In [21]:
product_features.to_csv(PRODUCT_PATH, index=False)
print(f'Updated: {PRODUCT_PATH}')
print(f'Final shape: {product_features.shape[0]:,} rows x {product_features.shape[1]} columns')

Updated: ../data/processed\product_features.csv
Final shape: 50 rows x 20 columns


---
## Section 17 — Feature Leakage Audit

**Rule:** A feature at date `t` must only use information available **before** `t`.

**Explicit checks:**
- Future demand → NOT used
- Future revenue → NOT used
- Future returns → NOT used
- Future cancellations → NOT used

In [22]:
leakage_audit = pd.DataFrame([
    {'Feature':'year',                  'Time Period':'Current date',   'Available?':'YES','Safe?':'✅ Safe'},
    {'Feature':'month_number',          'Time Period':'Current date',   'Available?':'YES','Safe?':'✅ Safe'},
    {'Feature':'quarter',               'Time Period':'Current date',   'Available?':'YES','Safe?':'✅ Safe'},
    {'Feature':'week_of_year',          'Time Period':'Current date',   'Available?':'YES','Safe?':'✅ Safe'},
    {'Feature':'day_of_week_num',       'Time Period':'Current date',   'Available?':'YES','Safe?':'✅ Safe'},
    {'Feature':'is_weekend',            'Time Period':'Current date',   'Available?':'YES','Safe?':'✅ Safe'},
    {'Feature':'is_month_start',        'Time Period':'Current date',   'Available?':'YES','Safe?':'✅ Safe'},
    {'Feature':'is_month_end',          'Time Period':'Current date',   'Available?':'YES','Safe?':'✅ Safe'},
    {'Feature':'lag_1',                 'Time Period':'t-1 day',        'Available?':'YES','Safe?':'✅ Safe'},
    {'Feature':'lag_7',                 'Time Period':'t-7 days',       'Available?':'YES','Safe?':'✅ Safe'},
    {'Feature':'lag_14',                'Time Period':'t-14 days',      'Available?':'YES','Safe?':'✅ Safe'},
    {'Feature':'lag_28',                'Time Period':'t-28 days',      'Available?':'YES','Safe?':'✅ Safe'},
    {'Feature':'rolling_mean_7',        'Time Period':'t-8 to t-2',     'Available?':'YES','Safe?':'✅ Safe'},
    {'Feature':'rolling_mean_14',       'Time Period':'t-15 to t-2',    'Available?':'YES','Safe?':'✅ Safe'},
    {'Feature':'rolling_mean_28',       'Time Period':'t-29 to t-2',    'Available?':'YES','Safe?':'✅ Safe'},
    {'Feature':'rolling_std_7',         'Time Period':'t-8 to t-2',     'Available?':'YES','Safe?':'✅ Safe'},
    {'Feature':'rolling_std_14',        'Time Period':'t-15 to t-2',    'Available?':'YES','Safe?':'✅ Safe'},
    {'Feature':'rolling_std_28',        'Time Period':'t-29 to t-2',    'Available?':'YES','Safe?':'✅ Safe'},
    {'Feature':'short_term_mean',       'Time Period':'Historical',     'Available?':'YES','Safe?':'✅ Safe'},
    {'Feature':'medium_term_mean',      'Time Period':'Historical',     'Available?':'YES','Safe?':'✅ Safe'},
    {'Feature':'long_term_mean',        'Time Period':'Historical',     'Available?':'YES','Safe?':'✅ Safe'},
    {'Feature':'short_vs_medium_growth','Time Period':'Historical',     'Available?':'YES','Safe?':'✅ Safe'},
    {'Feature':'medium_vs_long_growth', 'Time Period':'Historical',     'Available?':'YES','Safe?':'✅ Safe'},
    {'Feature':'demand_std_7',          'Time Period':'Historical',     'Available?':'YES','Safe?':'✅ Safe'},
    {'Feature':'demand_std_28',         'Time Period':'Historical',     'Available?':'YES','Safe?':'✅ Safe'},
    {'Feature':'cv_7',                  'Time Period':'Historical',     'Available?':'YES','Safe?':'✅ Safe'},
    {'Feature':'cv_28',                 'Time Period':'Historical',     'Available?':'YES','Safe?':'✅ Safe'},
    {'Feature':'rolling_return_rate_28','Time Period':'Historical',     'Available?':'YES','Safe?':'✅ Safe'},
    {'Feature':'rolling_cancel_rate_28','Time Period':'Historical',     'Available?':'YES','Safe?':'✅ Safe'},
    {'Feature':'rolling_revenue_mean_28','Time Period':'Historical',    'Available?':'YES','Safe?':'✅ Safe'},
    {'Feature':'revenue_growth_7_vs_28','Time Period':'Historical',     'Available?':'YES','Safe?':'✅ Safe'},
    {'Feature':'DailyAveragePrice',     'Time Period':'Current date t', 'Available?':'Conditional','Safe?':'⚠️ Conditional'},
    {'Feature':'DailyAverageDiscount',  'Time Period':'Current date t', 'Available?':'Conditional','Safe?':'⚠️ Conditional'},
])
safe_cnt = (leakage_audit['Safe?'].str.startswith('✅')).sum()
cond_cnt = (leakage_audit['Safe?'].str.startswith('⚠️')).sum()
print(f'Leakage Audit: ✅ Safe={safe_cnt}  ⚠️ Conditional={cond_cnt}  ❌ Leakage=0')
leakage_audit

Leakage Audit: ✅ Safe=31  ⚠️ Conditional=2  ❌ Leakage=0


,Feature,Time Period,Available?,Safe?
0,year,Current date,YES,✅ Safe
1,month_number,Current date,YES,✅ Safe
2,quarter,Current date,YES,✅ Safe
3,week_of_year,Current date,YES,✅ Safe
4,day_of_week_num,Current date,YES,✅ Safe
5,is_weekend,Current date,YES,✅ Safe
6,is_month_start,Current date,YES,✅ Safe
7,is_month_end,Current date,YES,✅ Safe
8,lag_1,t-1 day,YES,✅ Safe
9,lag_7,t-7 days,YES,✅ Safe


### Leakage Notes

1. All lag/rolling features safe — `shift(1)` excludes current day.
2. `DailyAveragePrice/Discount` conditional — safe if price known at prediction time.
3. `DailyRevenue, DailyOrderCount, DailyReturnCount, DailyCancellationCount` — target-adjacent, must NOT be model inputs for demand prediction.
4. **No future information leaks into any historical feature.**

---
## Section 18 — Feature Summary Table

In [23]:
feature_summary = pd.DataFrame([
    {'Feature':'year',                   'DataType':'int',   'Granularity':'Date',          'Purpose':'Demand forecasting / seasonal'},
    {'Feature':'month_number',           'DataType':'int',   'Granularity':'Date',          'Purpose':'Demand forecasting / seasonal'},
    {'Feature':'quarter',                'DataType':'int',   'Granularity':'Date',          'Purpose':'Seasonal demand patterns'},
    {'Feature':'week_of_year',           'DataType':'int',   'Granularity':'Date',          'Purpose':'Weekly demand patterns'},
    {'Feature':'day_of_week_num',        'DataType':'int',   'Granularity':'Date',          'Purpose':'Weekday demand patterns'},
    {'Feature':'is_weekend',             'DataType':'int',   'Granularity':'Date',          'Purpose':'Weekend demand effect'},
    {'Feature':'is_month_start',         'DataType':'int',   'Granularity':'Date',          'Purpose':'Month-start demand spike'},
    {'Feature':'is_month_end',           'DataType':'int',   'Granularity':'Date',          'Purpose':'Month-end demand spike'},
    {'Feature':'lag_1',                  'DataType':'float', 'Granularity':'ProductID x Date','Purpose':'Demand forecasting'},
    {'Feature':'lag_7',                  'DataType':'float', 'Granularity':'ProductID x Date','Purpose':'Weekly cycle forecasting'},
    {'Feature':'lag_14',                 'DataType':'float', 'Granularity':'ProductID x Date','Purpose':'Biweekly pattern detection'},
    {'Feature':'lag_28',                 'DataType':'float', 'Granularity':'ProductID x Date','Purpose':'Monthly cycle forecasting'},
    {'Feature':'rolling_mean_7',         'DataType':'float', 'Granularity':'ProductID x Date','Purpose':'Smoothed demand forecast input'},
    {'Feature':'rolling_mean_14',        'DataType':'float', 'Granularity':'ProductID x Date','Purpose':'Smoothed demand forecast input'},
    {'Feature':'rolling_mean_28',        'DataType':'float', 'Granularity':'ProductID x Date','Purpose':'Smoothed demand forecast input'},
    {'Feature':'rolling_std_7',          'DataType':'float', 'Granularity':'ProductID x Date','Purpose':'Short-term volatility'},
    {'Feature':'rolling_std_14',         'DataType':'float', 'Granularity':'ProductID x Date','Purpose':'Medium-term volatility'},
    {'Feature':'rolling_std_28',         'DataType':'float', 'Granularity':'ProductID x Date','Purpose':'Medium-term volatility'},
    {'Feature':'short_term_mean',        'DataType':'float', 'Granularity':'ProductID x Date','Purpose':'Temporal conflict signal'},
    {'Feature':'medium_term_mean',       'DataType':'float', 'Granularity':'ProductID x Date','Purpose':'Temporal conflict signal'},
    {'Feature':'long_term_mean',         'DataType':'float', 'Granularity':'ProductID x Date','Purpose':'Temporal conflict signal'},
    {'Feature':'short_vs_medium_growth', 'DataType':'float', 'Granularity':'ProductID x Date','Purpose':'Temporal conflict detection'},
    {'Feature':'medium_vs_long_growth',  'DataType':'float', 'Granularity':'ProductID x Date','Purpose':'Temporal conflict detection'},
    {'Feature':'demand_std_7',           'DataType':'float', 'Granularity':'ProductID x Date','Purpose':'Volatility detection'},
    {'Feature':'demand_std_28',          'DataType':'float', 'Granularity':'ProductID x Date','Purpose':'Volatility detection'},
    {'Feature':'cv_7',                   'DataType':'float', 'Granularity':'ProductID x Date','Purpose':'Growth/volatility conflict'},
    {'Feature':'cv_28',                  'DataType':'float', 'Granularity':'ProductID x Date','Purpose':'Growth/volatility conflict'},
    {'Feature':'rolling_return_rate_28', 'DataType':'float', 'Granularity':'ProductID x Date','Purpose':'Demand/quality conflict'},
    {'Feature':'rolling_cancel_rate_28', 'DataType':'float', 'Granularity':'ProductID x Date','Purpose':'Demand/cancellation conflict'},
    {'Feature':'rolling_revenue_mean_28','DataType':'float', 'Granularity':'ProductID x Date','Purpose':'Revenue signal'},
    {'Feature':'revenue_growth_7_vs_28', 'DataType':'float', 'Granularity':'ProductID x Date','Purpose':'Customer-revenue conflict'},
    {'Feature':'Recency',                'DataType':'int',   'Granularity':'CustomerID',      'Purpose':'Customer segmentation (RFM)'},
    {'Feature':'Frequency',              'DataType':'int',   'Granularity':'CustomerID',      'Purpose':'Customer segmentation (RFM)'},
    {'Feature':'Monetary',               'DataType':'float', 'Granularity':'CustomerID',      'Purpose':'Customer segmentation (RFM)'},
    {'Feature':'ReturnRate (cust)',       'DataType':'float', 'Granularity':'CustomerID',      'Purpose':'Customer quality signal'},
    {'Feature':'RecentDemand',           'DataType':'float', 'Granularity':'ProductID',       'Purpose':'Product intelligence'},
    {'Feature':'HistoricalAverageDemand','DataType':'float', 'Granularity':'ProductID',       'Purpose':'Product intelligence'},
    {'Feature':'DemandVolatility',       'DataType':'float', 'Granularity':'ProductID',       'Purpose':'Product intelligence / conflict'},
    {'Feature':'DemandGrowth',           'DataType':'float', 'Granularity':'ProductID',       'Purpose':'Product intelligence / conflict'},
    {'Feature':'repeat_customer_ratio',  'DataType':'float', 'Granularity':'ProductID',       'Purpose':'Customer-revenue conflict'},
])
print(f'Feature summary: {len(feature_summary)} features')
feature_summary

Feature summary: 40 features


,Feature,DataType,Granularity,Purpose
0,year,int,Date,Demand forecasting / seasonal
1,month_number,int,Date,Demand forecasting / seasonal
2,quarter,int,Date,Seasonal demand patterns
3,week_of_year,int,Date,Weekly demand patterns
4,day_of_week_num,int,Date,Weekday demand patterns
5,is_weekend,int,Date,Weekend demand effect
6,is_month_start,int,Date,Month-start demand spike
7,is_month_end,int,Date,Month-end demand spike
8,lag_1,float,ProductID x Date,Demand forecasting
9,lag_7,float,ProductID x Date,Weekly cycle forecasting


---
## Section 19 — Feature Quality Check

Quality check for all four datasets. Rows are **not deleted** — issues reported only.

In [24]:
def quality_report(df_check, name):
    print('='*60)
    print(f'  Quality Report: {name}')
    print('='*60)
    print(f'  Shape          : {df_check.shape[0]:,} rows x {df_check.shape[1]} cols')
    missing = df_check.isnull().sum()
    n_miss = (missing>0).sum()
    tot = missing.sum()
    print(f'  Missing values : {tot:,} total | {n_miss} cols')
    if tot > 0:
        for col, cnt in missing[missing>0].sort_values(ascending=False).head(10).items():
            print(f'    {col}: {cnt:,} ({cnt/len(df_check)*100:.2f}%)')
    print(f'  Duplicate rows : {df_check.duplicated().sum():,}')
    num = df_check.select_dtypes(include=[np.number])
    print(f'  Infinite values: {np.isinf(num).sum().sum():,}')
    print(f'  Negative values: {(num<0).sum().sum():,} (may be valid for growth cols)')
    print(f'  Data types     :', dict(df_check.dtypes.value_counts()))
    print()

quality_report(df, 'retail_cleaned.csv')
quality_report(demand_full, 'daily_product_demand.csv')
quality_report(customer_features, 'customer_features.csv')
quality_report(product_features, 'product_features.csv')

  Quality Report: retail_cleaned.csv
  Shape          : 100,000 rows x 27 cols
  Missing values : 0 total | 0 cols


  Duplicate rows : 0
  Infinite values: 0
  Negative values: 0 (may be valid for growth cols)
  Data types     : {<StringDtype(storage='python', na_value=nan)>: np.int64(13), dtype('int64'): np.int64(6), dtype('float64'): np.int64(6), dtype('<M8[us]'): np.int64(1), dtype('<M8[s]'): np.int64(1)}

  Quality Report: daily_product_demand.csv
  Shape          : 91,250 rows x 48 cols
  Missing values : 30,096 total | 24 cols
    long_term_mean: 4,500 (4.93%)
    medium_vs_long_growth: 4,500 (4.93%)
    lag_28: 1,400 (1.53%)
    cv_28: 1,400 (1.53%)
    rolling_std_28: 1,400 (1.53%)
    rolling_mean_28: 1,400 (1.53%)
    medium_term_mean: 1,400 (1.53%)
    short_vs_medium_growth: 1,400 (1.53%)
    rolling_revenue_std_28: 1,400 (1.53%)
    rolling_revenue_mean_28: 1,400 (1.53%)
  Duplicate rows : 0
  Infinite values: 0
  Negative values: 136,779 (may be valid for growth cols)
  Data types     : {dtype('float64'): np.int64(34), dtype('int32'): np.int64(5), <StringDtype(storage='python', na_valu

  Duplicate rows : 0
  Infinite values: 0
  Negative values: 0 (may be valid for growth cols)
  Data types     : {dtype('int64'): np.int64(6), dtype('float64'): np.int64(6), dtype('<M8[us]'): np.int64(2), <StringDtype(storage='python', na_value=nan)>: np.int64(1)}

  Quality Report: product_features.csv
  Shape          : 50 rows x 20 cols
  Missing values : 0 total | 0 cols
  Duplicate rows : 0
  Infinite values: 0
  Negative values: 28 (may be valid for growth cols)
  Data types     : {dtype('float64'): np.int64(11), dtype('int64'): np.int64(7), <StringDtype(storage='python', na_value=nan)>: np.int64(2)}



In [25]:
print('=== Expected NaN in daily_product_demand.csv ===')
print('  lag_1    : first 1  day  per product (50 x 1   =    50 rows)')
print('  lag_7    : first 7  days per product (50 x 7   =   350 rows)')
print('  lag_14   : first 14 days per product (50 x 14  =   700 rows)')
print('  lag_28   : first 28 days per product (50 x 28  = 1,400 rows)')
print('  rolling_mean/std (7d) : first 7 days per product')
print('  rolling_mean/std (28d): first 28 days per product')
print('  long_term_mean (90d)  : first 90 days per product (50 x 90 = 4,500 rows)')
print('  All NaN values CORRECT - insufficient history, not a data error.')
print()
check_cols = ['DailyQuantity','lag_1','rolling_mean_7','rolling_mean_28','short_vs_medium_growth','cv_7']
demand_full[check_cols].describe()

=== Expected NaN in daily_product_demand.csv ===
  lag_1    : first 1  day  per product (50 x 1   =    50 rows)
  lag_7    : first 7  days per product (50 x 7   =   350 rows)
  lag_14   : first 14 days per product (50 x 14  =   700 rows)
  lag_28   : first 28 days per product (50 x 28  = 1,400 rows)
  rolling_mean/std (7d) : first 7 days per product
  rolling_mean/std (28d): first 28 days per product
  long_term_mean (90d)  : first 90 days per product (50 x 90 = 4,500 rows)
  All NaN values CORRECT - insufficient history, not a data error.



,DailyQuantity,lag_1,rolling_mean_7,rolling_mean_28,short_vs_medium_growth,cv_7
count,91250.0000,91200.0000,90900.0000,89850.0000,89850.0000,90854.0000
mean,3.2892,3.2897,3.2894,3.2891,-0.0002,1.0779
std,3.4707,3.4711,1.3156,0.6544,0.3524,0.3620
min,0.0000,0.0000,0.0000,1.1071,-1.0000,0.1207
25%,0.0000,0.0000,2.2857,2.8214,-0.2444,0.8273
50%,3.0000,3.0000,3.1429,3.2857,-0.0145,1.0281
75%,5.0000,5.0000,4.1429,3.7143,0.2308,1.2689
max,31.0000,31.0000,10.5714,5.9643,1.7586,2.6458


---
## Section 20 — Conflict-Aware Intelligence Readiness

Verify all five planned conflict signals have required features available.

> Conflict detection NOT implemented here. Readiness check only.

In [26]:
print('='*65)
print('  CONFLICT-AWARE INTELLIGENCE READINESS CHECK')
print('='*65)
conflicts = [
    {'name':'1. Temporal Conflict','description':'Short-term demand UP | Long-term demand DOWN',
     'features':['short_term_mean','medium_term_mean','long_term_mean','short_vs_medium_growth','medium_vs_long_growth']},
    {'name':'2. Commercial / Customer Conflict','description':'Revenue UP | Unique customers DOWN',
     'features':['rolling_revenue_mean_28','revenue_growth_7_vs_28','repeat_customer_ratio','UniqueCustomers']},
    {'name':'3. Demand / Quality Conflict','description':'Demand UP | Return rate UP',
     'features':['rolling_mean_28','rolling_return_rate_28','daily_return_rate']},
    {'name':'4. Demand / Cancellation Conflict','description':'Demand UP | Cancellation rate UP',
     'features':['rolling_mean_28','rolling_cancel_rate_28','daily_cancel_rate']},
    {'name':'5. Growth / Volatility Conflict','description':'Demand growth UP | Demand volatility UP',
     'features':['short_vs_medium_growth','medium_vs_long_growth','cv_7','cv_28','demand_std_7','demand_std_28']},
]
all_ready = True
for c in conflicts:
    print(f"\n  {c['name']}")
    print(f"  Scenario: {c['description']}")
    cr = True
    for feat in c['features']:
        avail = feat in demand_full.columns or feat in product_features.columns or feat in customer_features.columns
        if not avail: cr = False; all_ready = False
        print(f"    {'✅ Available' if avail else '❌ MISSING'} - {feat}")
    print(f"  Verdict: {'✅ READY' if cr else '❌ NOT READY'}")
print(f"\n{'='*65}")
print(f"  {'✅ ALL SIGNALS READY' if all_ready else '⚠️ SOME FEATURES MISSING'}")
print('='*65)

  CONFLICT-AWARE INTELLIGENCE READINESS CHECK

  1. Temporal Conflict
  Scenario: Short-term demand UP | Long-term demand DOWN
    ✅ Available - short_term_mean
    ✅ Available - medium_term_mean
    ✅ Available - long_term_mean
    ✅ Available - short_vs_medium_growth
    ✅ Available - medium_vs_long_growth
  Verdict: ✅ READY

  2. Commercial / Customer Conflict
  Scenario: Revenue UP | Unique customers DOWN
    ✅ Available - rolling_revenue_mean_28
    ✅ Available - revenue_growth_7_vs_28
    ✅ Available - repeat_customer_ratio
    ✅ Available - UniqueCustomers
  Verdict: ✅ READY

  3. Demand / Quality Conflict
  Scenario: Demand UP | Return rate UP
    ✅ Available - rolling_mean_28
    ✅ Available - rolling_return_rate_28
    ✅ Available - daily_return_rate
  Verdict: ✅ READY

  4. Demand / Cancellation Conflict
  Scenario: Demand UP | Cancellation rate UP
    ✅ Available - rolling_mean_28
    ✅ Available - rolling_cancel_rate_28
    ✅ Available - daily_cancel_rate
  Verdict: ✅ READ

---
## Section 21 — Final Feature Engineering Summary

In [27]:
print('='*65)
print('  FEATURE ENGINEERING SUMMARY - RetailMind AI Phase 3')
print('='*65)
print()
print('  Feature Group Counts')
print('  Temporal features            : 11')
print('  Lag features                 : 4')
print('  Rolling mean/std features    : 6')
print('  Trend features               : 5')
print('  Volatility features          : 4')
print('  Return/cancellation signals  : 4')
print('  Revenue signals              : 5')
print(f'  Customer features            : {customer_features.shape[1]-1}')
print(f'  Product features             : {product_features.shape[1]-4}')
print()
print('  Saved Datasets')
print(f'  retail_cleaned.csv        : {df.shape[0]:,} x {df.shape[1]}')
print(f'  daily_product_demand.csv  : {demand_full.shape[0]:,} x {demand_full.shape[1]}')
print(f'  customer_features.csv     : {customer_features.shape[0]:,} x {customer_features.shape[1]}')
print(f'  product_features.csv      : {product_features.shape[0]:,} x {product_features.shape[1]}')
print()
print('  Feature Usage by Goal')
print('  1. DEMAND FORECASTING: lag+rolling+temporal+trend+volatility | Target: DailyQuantity')
print('  2. CUSTOMER SEGMENTATION: RFM + rates + discount + frequency')
print('  3. PRODUCT INTELLIGENCE: qty/revenue/return/cancel + growth + volatility')
print('  4. CONFLICT DETECTION (Phase 4+):')
print('     Temporal   : short_vs_medium_growth + medium_vs_long_growth')
print('     Commercial : revenue_growth_7_vs_28 + repeat_customer_ratio')
print('     Quality    : rolling_mean_28 + rolling_return_rate_28')
print('     Cancel     : rolling_mean_28 + rolling_cancel_rate_28')
print('     Volatility : short_vs_medium_growth + cv_7/cv_28')
print()
print('='*65)
print('  STATUS: PHASE 3 COMPLETE')
print('  NEXT  : PHASE 4 - DEMAND FORECASTING MODEL DEVELOPMENT')
print('  STOP  : Do not implement models in this notebook.')
print('='*65)

  FEATURE ENGINEERING SUMMARY - RetailMind AI Phase 3

  Feature Group Counts
  Temporal features            : 11
  Lag features                 : 4
  Rolling mean/std features    : 6
  Trend features               : 5
  Volatility features          : 4
  Return/cancellation signals  : 4
  Revenue signals              : 5
  Customer features            : 14
  Product features             : 16

  Saved Datasets
  retail_cleaned.csv        : 100,000 x 27
  daily_product_demand.csv  : 91,250 x 48
  customer_features.csv     : 43,233 x 15
  product_features.csv      : 50 x 20

  Feature Usage by Goal
  1. DEMAND FORECASTING: lag+rolling+temporal+trend+volatility | Target: DailyQuantity
  2. CUSTOMER SEGMENTATION: RFM + rates + discount + frequency
  3. PRODUCT INTELLIGENCE: qty/revenue/return/cancel + growth + volatility
  4. CONFLICT DETECTION (Phase 4+):
     Temporal   : short_vs_medium_growth + medium_vs_long_growth
     Commercial : revenue_growth_7_vs_28 + repeat_customer_ratio
     

---
## End of Notebook

**Next Steps — Phase 4: Demand Forecasting**

1. Load `data/processed/daily_product_demand.csv`
2. Define a **time-based** train/validation/test split (never random shuffle)
3. Build demand forecasting models (XGBoost, LightGBM, or LSTM)
4. Target: `DailyQuantity` per ProductID per Date
5. Evaluate with time-series metrics: MAE, RMSE, MAPE

**Do not implement conflict detection until forecasting is validated.**